# Gecikmeli iadelerin beş yıllık hikâyesi
Yakup Kerim TİLKİ · 2516016032 · 16 Ekim 2026

**Soru:** SAÜ Kütüphane Veri Seti’nde 2020–2024 yılları arasında gecikmeli iade oranı kitap kategorilerine ve kütüphanelere göre nasıl değişmektedir?

Birim ödünç işlemidir, öğrenci değildir. Veri üreticisi ve açık kullanım lisansı doğrulanmadığından sonuçlar yalnızca sağlanan dosyanın kayıtlarını betimler; SAÜ'nün gerçek performansı hakkında genellenmez.

## 1 Veri yükleme
Ham Excel'in öğrenci adları ve numaraları analiz edilmedi. `islemler_kimliksiz.csv.gz`, işlem tablosunun kitap-kategori ve kütüphane tablolarıyla çoktan-bire birleştirilmiş, kimlik sütunları çıkarılmış analiz girdisidir. Kaynaktaki işlem anahtarlarının benzersizliği hazırlıkta kontrol edildi. Tekrarlanan kimliksiz satırları silmiyoruz: farklı gerçek işlemler aynı gözlemlenebilir değerlere sahip olabilir. Orijinal dosya özeti ve SHA-256 izi `kaynak.json` içindedir.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
ROOT = Path.cwd() if (Path.cwd() / 'data/raw').exists() else Path.cwd().parent
raw = pd.read_csv(ROOT / 'data/raw/islemler_kimliksiz.csv.gz')
source = json.loads((ROOT / 'data/raw/kaynak.json').read_text(encoding='utf-8'))
print('Girdi satırı:', len(raw))
print('Sütunlar:', ', '.join(raw.columns))
print('Eksik hücre:', int(raw.isna().sum().sum()))


Girdi satırı: 15000
Sütunlar: OduncTarihi, IadeTarihi, BeklenenIadeTarihi, SuresiGectiMi, GecSuresi, OduncSuresi, KategoriAdi, AnaKategori, KutuphaneAdi
Eksik hücre: 0


## 2 Temizleme ve doğrulama
Tarihler açık ISO biçimiyle okunur. İade veya vade tarihi eksik olan, iadesi ödünç tarihinden önce olan ya da dönemin dışında kalan kayıtlar oran paydasına alınmaz. Sayımlar denetim çıktısında saklanır. Gecikme, gerçek iade tarihinin beklenen iade tarihinden **sonra** olmasıdır; vade gününde iade gecikme sayılmaz. Gecikme gününü tarihlerden yeniden hesaplayıp hazır bayrak ve sürelerle karşılaştırıyoruz.

Yıl, **ödünç alma yılıdır**. 2024'te alınan ve Ocak 2025'te iade edilen kitaplar 2024 grubundadır. Kayıtların tamamında iade tarihi vardır; gözlenmeyen/açık ödünçlere ilişkin çıkarım yapılmaz. Aykırı uzun gecikmeler geçerli tarihlerden geldiği için tutulur. Kategorilerde yalnızca Türkçe gösterim adları düzenlenir.


In [2]:
df = raw.copy()
for c in ['OduncTarihi','IadeTarihi','BeklenenIadeTarihi']:
    df[c] = pd.to_datetime(df[c], format='%Y-%m-%d', errors='coerce')
invalid = df[['OduncTarihi','IadeTarihi','BeklenenIadeTarihi','KategoriAdi','KutuphaneAdi']].isna().any(axis=1)
invalid |= df.IadeTarihi.lt(df.OduncTarihi) | df.BeklenenIadeTarihi.lt(df.OduncTarihi)
outside = ~df.OduncTarihi.dt.year.between(2020,2024)
audit = {'input_rows':len(df),'invalid_rows':int(invalid.sum()),'outside_period':int(outside.sum())}
df = df.loc[~invalid & ~outside].copy()
df['year'] = df.OduncTarihi.dt.year.astype(int)
df['days'] = (df.IadeTarihi-df.BeklenenIadeTarihi).dt.days.clip(lower=0)
df['late'] = df.days.gt(0)
audit['flag_mismatch'] = int(df.late.ne(df.SuresiGectiMi.eq('Evet')).sum())
audit['days_mismatch'] = int(df.days.ne(df.GecSuresi).sum())
audit['duration_mismatch'] = int((df.IadeTarihi-df.OduncTarihi).dt.days.ne(df.OduncSuresi).sum())
audit['retained_rows'] = len(df)
audit['excluded_rows'] = len(raw)-len(df)
audit['returns_after_2024'] = int(df.IadeTarihi.gt('2024-12-31').sum())
name_map={'Tip':'Tıp','Siir':'Şiir','Hikaye':'Hikâye','Egitim':'Eğitim','Muhendislik':'Mühendislik','Isletme':'İşletme','Merkez Kutuphane':'Merkez','Muhendislik Kutuphanesi':'Mühendislik','Tip Kutuphanesi':'Tıp'}
df['category']=df.KategoriAdi.replace(name_map)
df['library']=df.KutuphaneAdi.replace(name_map)
print(json.dumps(audit,ensure_ascii=False,indent=2))
assert audit['flag_mismatch']==0 and audit['days_mismatch']==0


{
  "input_rows": 15000,
  "invalid_rows": 0,
  "outside_period": 0,
  "flag_mismatch": 0,
  "days_mismatch": 0,
  "duration_mismatch": 0,
  "retained_rows": 15000,
  "excluded_rows": 0,
  "returns_after_2024": 191
}


## 3 Keşifsel analiz
Oran = geciken iade sayısı / gözlenen tüm iadeler × 100. Grup oranlarının basit ortalaması yerine işlem sayıları üzerinden ağırlıklı toplam kullanılır. Küçük grupların oynaklığına dikkat etmek için örneklem sayısı ve Wilson %95 aralığı verilir. Bu aralıklar bağımsız işlemler varsayar; aynı öğrencinin birden fazla işlemi ve veri üretim sürecinin belirsizliği nedeniyle yalnızca betimleyici yardımcı göstergedir, kurumlar arasında kesin fark kanıtı değildir.


In [3]:
def aggregate(keys):
    g=df.groupby(keys,observed=True).agg(n=('late','size'),late=('late','sum')).reset_index()
    p=g['late']/g.n; z=1.96; den=1+z*z/g.n
    center=(p+z*z/(2*g.n))/den
    half=z*np.sqrt(p*(1-p)/g.n+z*z/(4*g.n*g.n))/den
    g['rate']=p*100; g['lo']=(center-half)*100; g['hi']=(center+half)*100
    return g
annual=aggregate(['year'])
libraries=aggregate(['library']).sort_values('rate',ascending=False)
categories=aggregate(['category']).sort_values('rate',ascending=False)
annual_library=aggregate(['year','library'])
annual_category=aggregate(['year','category'])
cube=aggregate(['year','library','category'])
print('Yıllar\n',annual.to_string(index=False))
print('Kütüphaneler\n',libraries.to_string(index=False))
print('Kategoriler\n',categories.to_string(index=False))


Yıllar
  year    n  late      rate        lo        hi
 2020 1821   790 43.382757 41.122718 45.670656
 2021 2653  1203 45.344893 43.458602 47.244645
 2022 3423  1547 45.194274 43.533312 46.866011
 2023 3444  1600 46.457607 44.796759 48.126350
 2024 3659  1567 42.825909 41.230908 44.435958
Kütüphaneler
     library    n  late      rate        lo        hi
Mühendislik 4475  2024 45.229050 43.775474 46.690811
     Merkez 9015  4020 44.592346 43.568769 45.620530
        Tıp 1510   663 43.907285 41.422713 46.422779
Kategoriler
          category    n  late      rate        lo        hi
         Biyoloji  619   295 47.657512 43.749459 51.594461
              Tıp  612   289 47.222222 43.296570 51.182529
        Matematik 1811   854 47.156267 44.865585 49.458989
              Dil  225   105 46.666667 40.258531 53.186717
            Roman  288   134 46.527778 40.850578 52.296389
            Tarih  708   323 45.621469 41.986019 49.304178
          Felsefe  407   184 45.208845 40.440770 50.066521

## 4 Bulgular ve yorum sınırları
1. Gecikmeler dosyadaki işlemlerin yaklaşık %45'ini kapsıyor.
2. 2024 oranı 2023'e göre düşüyor; beş yıl boyunca kesintisiz bir iyileşme yok.
3. Kütüphaneler arasındaki toplam oran farkı küçük. Merkez'in geciken işlem sayısının yüksek olması daha büyük işlem hacmiyle birlikte okunmalı.
4. Kategoriler arasında fark var; işlem sayıları ve belirsizlik aralıkları olmadan bir sıralama başarı/başarısızlık diye yorumlanamaz.
5. Geciken iadelerin ortanca gecikmesi beş gün. Ortanca, gecikmeler küçükten büyüğe sıralandığında ortadaki değerdir.

Bunlar dosyayı betimleyen bulgulardır. Kitabın zorluğu, sınav dönemi, hatırlatma sistemi veya kütüphane politikası hakkında nedensel açıklama yapılamaz. Metin taslağı yapay zekâ desteğiyle hazırlanmıştır; öğrenci kendi yorumunu gözden geçirip ifade etmelidir.


In [4]:
late_df=df[df.late]
bins=[0,3,7,14,100000]
labels=['1–3 gün','4–7 gün','8–14 gün','15+ gün']
binned=pd.cut(late_df.days,bins=bins,labels=labels)
hist=binned.value_counts(sort=False).rename_axis('label').reset_index(name='n')
hist['share']=hist.n/len(late_df)*100
facts={'n':len(df),'late':int(df.late.sum()),'rate':float(df.late.mean()*100),'median_days':float(late_df.days.median()),'max_days':int(late_df.days.max()),'mean_days':float(late_df.days.mean()),'year_change_pp':float(annual.iloc[-1].rate-annual.iloc[-2].rate),'library_gap_pp':float(libraries.rate.max()-libraries.rate.min()),'category_gap_pp':float(categories.rate.max()-categories.rate.min())}
print(json.dumps(facts,ensure_ascii=False,indent=2))
print(hist.to_string(index=False))
assert annual.n.sum()==len(df)==cube.n.sum()
assert annual.late.sum()==df.late.sum()==hist.n.sum()
assert not any(c in df.columns for c in ['AdSoyad','OgrenciNo','OgrenciID'])


{
  "n": 15000,
  "late": 6707,
  "rate": 44.71333333333333,
  "median_days": 5.0,
  "max_days": 26,
  "mean_days": 5.791710153570896,
  "year_change_pp": -3.6316987149881683,
  "library_gap_pp": 1.3217655111176896,
  "category_gap_pp": 5.650077172078717
}
   label    n     share
 1–3 gün 2419 36.066796
 4–7 gün 2348 35.008200
8–14 gün 1686 25.137916
 15+ gün  254  3.787088


## 5 Web için özet dosyaları
Siteye yalnızca yıl-kütüphane-kategori düzeyinde sayımlar, oranlar ve gecikme dağılımı aktarılır. İşlem satırları veya kişi bilgileri aktarılmaz. Özet CSV'ler analizi denetlemek içindir; `summary.json` grafiklerin tek veri kaynağıdır. Görselleştirme kodunda ölçüm değerleri elle yazılmaz.


In [5]:
tables={'annual':annual,'libraries':libraries,'categories':categories,'annual_library':annual_library,'annual_category':annual_category,'cube':cube,'histogram':hist}
summary={'meta':{'period':'2020–2024','year_basis':'Ödünç yılı','source':'SAÜ Kütüphane Veri Seti · sağlanan dosya','accessed':'2026-10-16','license':source['license']},'facts':facts,'audit':audit}
for key,table in tables.items():
    summary[key]=json.loads(table.to_json(orient='records',force_ascii=False,double_precision=8))
    table.to_csv(ROOT/'data/processed'/f'{key}.csv',index=False,encoding='utf-8')
for target in [ROOT/'data/processed/summary.json',ROOT/'site/data/summary.json']:
    target.write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
print('Web veri boyutu:',(ROOT/'site/data/summary.json').stat().st_size,'bayt')
assert (ROOT/'site/data/summary.json').stat().st_size < 5_000_000
print('Tüm doğrulamalar geçti.')


Web veri boyutu: 88733 bayt
Tüm doğrulamalar geçti.
